# 📖 Lab 2: Stream Videos

Our second functional requirement: **Users can watch (stream) videos.**

Once a video is uploaded and processed (Lab 1), users need to watch it. The naive approach — download the entire file — fails for large videos. The production approach uses **adaptive bitrate streaming** with manifest files and segments.

## 🏗️ Architecture — Before (Lab 1: Upload Only)

```
┌────────┐  POST /presigned_url  ┌───────────────┐       ┌──────────────┐
│ Client │─────────────────────>│ Video Service │──────>│  Metadata DB │
│        │  PUT via presigned   └───────────────┘       └──────────────┘
│        │─────────────────────> S3 (raw video)
└────────┘
```

## 🏗️ Architecture — After (Upload + Streaming)

```
┌────────┐  GET /videos/:videoId  ┌───────────────┐  query   ┌──────────────┐
│        │──────────────────────>│ Video Service │────────>│ Metadata DB  │
│ Client │                       │               │         │ (manifestUrl)│
│        │<─────────────────── │               │         └──────────────┘
│        │ {metadata, manifestUrl}└──────────────┘
│        │
│        │  1. GET master.m3u8 (manifest)
│        │  2. Choose quality (720p, 480p...)
│        │  3. GET segments sequentially
│        │  4. Adapt quality on bandwidth change
│        │
│        │──download segments──> S3 / CDN (processed segments)
└────────┘                       /processed/{videoId}/720p/seg_001.ts
                                                         /seg_002.ts ...
```

## 3 Approaches to Watching

| # | Approach | User Experience |
|---|----------|----------------|
| ❌ | **Full download** | Wait 13+ min for 10GB file, then play |
| 🟡 | **Segment streaming** (fixed quality) | Fast start, but buffers if bandwidth drops |
| ✅ | **Adaptive bitrate streaming** | Fast start, seamless quality switching, no buffering |

## Learning Objectives

- Understand why full download fails for video streaming
- See how HLS manifest files work (master + media manifests)
- Build segments and manifests from scratch, store them in S3
- Simulate adaptive bitrate streaming — the client switching quality mid-stream
- Compare all three approaches side-by-side

## 🛠️ Setup

Make sure MinIO + PostgreSQL are running from Lab 1:

```bash
cd 06-system-designs/youtube
docker compose up -d
```

Select the **"YouTube (Python)"** kernel.

In [ ]:
import psycopg2
import psycopg2.extras
from minio import Minio
import time
import os
import io
import json

DB_CONFIG = {
    "host": "localhost",
    "port": 5434,
    "user": "demo",
    "password": "demo",
    "database": "youtube",
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

minio_client = Minio(
    "localhost:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False,
)

conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM videos")
print(f"✅ PostgreSQL: {cur.fetchone()[0]} videos")
cur.close()
conn.close()
print(f"✅ MinIO: {[b.name for b in minio_client.list_buckets()]}")

## 🎬 Simulating Processed Video

In production, the Video Processing Pipeline (Lab 1) would split and transcode the video. For this lab, we'll **simulate** what that pipeline produces: segments in multiple quality levels + HLS manifest files.

We'll create fake segments (random bytes representing video data) and real HLS `.m3u8` manifest files.

In [ ]:
VIDEO_ID = "dQw4w9WgXcQ"
SEGMENT_DURATION = 5  # seconds per segment
TOTAL_DURATION = 30   # simulate 30 seconds of video
NUM_SEGMENTS = TOTAL_DURATION // SEGMENT_DURATION

# Quality levels: resolution → simulated segment size
QUALITIES = {
    "1080p": {"bandwidth": 5000000, "resolution": "1920x1080", "segment_kb": 3000},
    "720p":  {"bandwidth": 2500000, "resolution": "1280x720",  "segment_kb": 1500},
    "480p":  {"bandwidth": 1000000, "resolution": "854x480",   "segment_kb": 600},
    "360p":  {"bandwidth": 500000,  "resolution": "640x360",   "segment_kb": 300},
}

BUCKET = "processed-videos"

# Create fake segments for each quality level
print(f"🎬 Generating simulated video segments for '{VIDEO_ID}'")
print(f"   Duration: {TOTAL_DURATION}s | Segments: {NUM_SEGMENTS} × {SEGMENT_DURATION}s each\n")

for quality, info in QUALITIES.items():
    for seg_num in range(NUM_SEGMENTS):
        segment_data = os.urandom(info["segment_kb"] * 1024)
        object_name = f"{VIDEO_ID}/{quality}/segment_{seg_num:03d}.ts"
        minio_client.put_object(
            BUCKET, object_name, io.BytesIO(segment_data),
            length=len(segment_data), content_type="video/mp2t",
        )
    total_size = info["segment_kb"] * NUM_SEGMENTS
    print(f"  {quality}: {NUM_SEGMENTS} segments, ~{total_size}KB total")

print(f"\n✅ {NUM_SEGMENTS * len(QUALITIES)} segments created in S3")

## 📋 HLS Manifest Files

HLS (HTTP Live Streaming) uses `.m3u8` manifest files as an index for video segments. There are two levels:

1. **Master manifest** — lists all available quality levels
2. **Media manifest** (one per quality) — lists the segment URLs for that quality

The client reads the master manifest first, picks a quality, then reads that quality's media manifest to know which segments to download.

```
master.m3u8
  ├── 1080p.m3u8 → segment_000.ts, segment_001.ts, ...
  ├── 720p.m3u8  → segment_000.ts, segment_001.ts, ...
  ├── 480p.m3u8  → segment_000.ts, segment_001.ts, ...
  └── 360p.m3u8  → segment_000.ts, segment_001.ts, ...
```

In [ ]:
# Generate HLS manifest files
BASE_URL = f"http://localhost:9000/{BUCKET}/{VIDEO_ID}"

# Media manifests (one per quality)
for quality, info in QUALITIES.items():
    lines = ["#EXTM3U", "#EXT-X-VERSION:3", f"#EXT-X-TARGETDURATION:{SEGMENT_DURATION}", "#EXT-X-MEDIA-SEQUENCE:0"]
    for seg_num in range(NUM_SEGMENTS):
        lines.append(f"#EXTINF:{SEGMENT_DURATION}.000,")
        lines.append(f"{BASE_URL}/{quality}/segment_{seg_num:03d}.ts")
    lines.append("#EXT-X-ENDLIST")

    manifest_content = "\n".join(lines)
    object_name = f"{VIDEO_ID}/{quality}.m3u8"
    minio_client.put_object(
        BUCKET, object_name, io.BytesIO(manifest_content.encode()),
        length=len(manifest_content), content_type="application/vnd.apple.mpegurl",
    )

# Master manifest (lists all qualities)
master_lines = ["#EXTM3U"]
for quality, info in QUALITIES.items():
    master_lines.append(
        f'#EXT-X-STREAM-INF:BANDWIDTH={info["bandwidth"]},RESOLUTION={info["resolution"]}'
    )
    master_lines.append(f"{BASE_URL}/{quality}.m3u8")

master_content = "\n".join(master_lines)
minio_client.put_object(
    BUCKET, f"{VIDEO_ID}/master.m3u8", io.BytesIO(master_content.encode()),
    length=len(master_content), content_type="application/vnd.apple.mpegurl",
)

# Update video metadata with manifest URL
manifest_url = f"{BASE_URL}/master.m3u8"
conn = get_connection()
cur = conn.cursor()
cur.execute("UPDATE videos SET manifest_url = %s, status = 'ready' WHERE id = %s", (manifest_url, VIDEO_ID))
conn.commit()
cur.close()
conn.close()

print("📋 Generated HLS manifests:\n")
print("  Master manifest (master.m3u8):")
print(f"  {manifest_url}\n")
print("  Contents:")
for line in master_content.split("\n"):
    print(f"    {line}")

print(f"\n  Media manifest example (720p.m3u8):")
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT manifest_url FROM videos WHERE id = %s", (VIDEO_ID,))
print(f"  Stored in DB: manifest_url = {cur.fetchone()[0]}")
cur.close()
conn.close()

## ❌ Approach 1: Full Download

The naive approach: download the entire video file before playing. Let's see why this fails.

In [ ]:
def watch_full_download(video_id: str, quality: str = "720p") -> dict:
    """
    ❌ Full download: download ALL segments at once, then "play".
    User waits for the entire video before they can watch anything.
    """
    start = time.time()
    total_bytes = 0

    # Download every segment
    for seg_num in range(NUM_SEGMENTS):
        obj = minio_client.get_object(BUCKET, f"{video_id}/{quality}/segment_{seg_num:03d}.ts")
        data = obj.read()
        total_bytes += len(data)
        obj.close()

    download_time = (time.time() - start) * 1000

    return {
        "approach": "full_download",
        "quality": quality,
        "total_bytes": total_bytes,
        "download_ms": round(download_time),
        "time_to_first_frame_ms": round(download_time),  # Can't play until everything is downloaded
    }


result = watch_full_download(VIDEO_ID, "720p")
print(f"❌ Full Download Approach:\n")
print(f"  Quality: {result['quality']}")
print(f"  Downloaded: {result['total_bytes'] / 1024:.0f}KB ({NUM_SEGMENTS} segments)")
print(f"  Total download time: {result['download_ms']}ms")
print(f"  Time to first frame: {result['time_to_first_frame_ms']}ms ← must wait for EVERYTHING")
print(f"\n  ⚠️  User stares at a loading screen for {result['download_ms']}ms before seeing anything.")
print(f"     For a real 2-hour movie at 720p (~3GB): that's minutes of waiting.")

## 🟡 Approach 2: Segment Streaming (Fixed Quality)

Better: download segments one at a time. Start playing as soon as the first segment is ready, download the rest in the background.

In [ ]:
def watch_segment_streaming(video_id: str, quality: str = "720p") -> dict:
    """
    🟡 Segment streaming: download and play segments one at a time.
    Playback starts after the first segment loads.
    """
    segments_log = []
    total_bytes = 0

    for seg_num in range(NUM_SEGMENTS):
        start = time.time()
        obj = minio_client.get_object(BUCKET, f"{video_id}/{quality}/segment_{seg_num:03d}.ts")
        data = obj.read()
        obj.close()
        download_ms = (time.time() - start) * 1000

        total_bytes += len(data)
        segments_log.append({
            "segment": seg_num,
            "quality": quality,
            "size_kb": len(data) // 1024,
            "download_ms": round(download_ms),
        })

    return {
        "approach": "segment_streaming",
        "quality": quality,
        "total_bytes": total_bytes,
        "time_to_first_frame_ms": segments_log[0]["download_ms"],
        "segments": segments_log,
    }


result = watch_segment_streaming(VIDEO_ID, "720p")
print(f"🟡 Segment Streaming (fixed {result['quality']}):\n")
print(f"  Time to first frame: {result['time_to_first_frame_ms']}ms ← play starts after 1 segment!")
print(f"  Total data: {result['total_bytes'] / 1024:.0f}KB\n")

print(f"  {'Segment':<10} {'Quality':<10} {'Size':<10} {'Download'}")
print(f"  {'-'*40}")
for s in result["segments"]:
    marker = "▶️ PLAY" if s["segment"] == 0 else ""
    print(f"  seg_{s['segment']:03d}    {s['quality']:<10} {s['size_kb']}KB     {s['download_ms']}ms  {marker}")

print(f"\n  ✅ Playback starts after just {result['time_to_first_frame_ms']}ms (vs {watch_full_download(VIDEO_ID)['download_ms']}ms for full download)")
print(f"  ⚠️  But: if bandwidth drops mid-stream, 720p segments take longer → buffering!")

## ✅ Approach 3: Adaptive Bitrate Streaming (ABR)

The production approach. The client:
1. Downloads the **master manifest** → knows all available quality levels
2. Picks initial quality based on estimated bandwidth
3. Downloads segments one at a time
4. **Measures download speed** after each segment
5. If bandwidth drops → **switches to lower quality** (smaller segments, faster download)
6. If bandwidth improves → **switches to higher quality**

This is exactly what YouTube, Netflix, and every modern video player does. No buffering, seamless quality transitions.

Let's simulate fluctuating network conditions and watch the client adapt.

In [ ]:
import requests

def parse_master_manifest(manifest_url: str) -> list[dict]:
    """Download and parse the HLS master manifest to extract available qualities."""
    resp = requests.get(manifest_url)
    lines = resp.text.strip().split("\n")

    qualities = []
    for i, line in enumerate(lines):
        if line.startswith("#EXT-X-STREAM-INF"):
            # Parse bandwidth and resolution from the tag
            parts = {kv.split("=")[0]: kv.split("=")[1] for kv in line.split(":")[1].split(",")}
            qualities.append({
                "bandwidth": int(parts["BANDWIDTH"]),
                "resolution": parts["RESOLUTION"],
                "url": lines[i + 1],
            })
    return qualities


def choose_quality(qualities: list[dict], available_bandwidth_kbps: int) -> dict:
    """Pick the highest quality that fits within the available bandwidth."""
    available_bps = available_bandwidth_kbps * 1000
    # Sort by bandwidth descending, pick the first one that fits
    for q in sorted(qualities, key=lambda x: x["bandwidth"], reverse=True):
        if q["bandwidth"] <= available_bps:
            return q
    return qualities[-1]  # Fallback to lowest quality


def watch_adaptive(video_id: str, manifest_url: str, bandwidth_schedule: list[int]) -> dict:
    """
    ✅ Adaptive bitrate streaming: switches quality based on simulated bandwidth.
    
    bandwidth_schedule: list of bandwidth values (kbps) per segment.
    Simulates network conditions changing over time.
    """
    # Step 1: Fetch and parse master manifest
    qualities = parse_master_manifest(manifest_url)

    segments_log = []
    total_bytes = 0

    for seg_num in range(NUM_SEGMENTS):
        # Step 2: Pick quality based on current "bandwidth"
        current_bw = bandwidth_schedule[seg_num % len(bandwidth_schedule)]
        chosen = choose_quality(qualities, current_bw)

        # Extract quality name from URL (e.g., "720p" from ".../720p.m3u8")
        quality_name = chosen["url"].split("/")[-1].replace(".m3u8", "")

        # Step 3: Download the segment
        start = time.time()
        obj = minio_client.get_object(BUCKET, f"{video_id}/{quality_name}/segment_{seg_num:03d}.ts")
        data = obj.read()
        obj.close()
        download_ms = (time.time() - start) * 1000

        total_bytes += len(data)
        segments_log.append({
            "segment": seg_num,
            "quality": quality_name,
            "bandwidth_kbps": current_bw,
            "size_kb": len(data) // 1024,
            "download_ms": round(download_ms),
        })

    return {
        "approach": "adaptive_bitrate",
        "total_bytes": total_bytes,
        "time_to_first_frame_ms": segments_log[0]["download_ms"],
        "segments": segments_log,
    }


# Simulate: good bandwidth → drops → recovers
# This simulates a user on WiFi, walking to poor signal, then recovering
bandwidth_schedule = [5000, 5000, 800, 400, 400, 2000]

print("✅ Adaptive Bitrate Streaming — simulating fluctuating network:\n")
print(f"  Bandwidth schedule (kbps per segment): {bandwidth_schedule}\n")

result = watch_adaptive(VIDEO_ID, manifest_url, bandwidth_schedule)

print(f"  {'Seg':<6} {'Bandwidth':<12} {'Quality':<10} {'Size':<10} {'Download'}")
print(f"  {'─'*50}")
for s in result["segments"]:
    # Visual indicator for quality changes
    indicator = ""
    if s["segment"] > 0:
        prev = result["segments"][s["segment"] - 1]["quality"]
        if s["quality"] != prev:
            indicator = " ← QUALITY SWITCH"
    marker = " ▶️ PLAY" if s["segment"] == 0 else ""
    print(f"  {s['segment']:<6} {s['bandwidth_kbps']:<7} kbps  {s['quality']:<10} {s['size_kb']}KB     {s['download_ms']}ms{indicator}{marker}")

print(f"\n  📊 Total data: {result['total_bytes'] / 1024:.0f}KB")
print(f"  ⏱️  Time to first frame: {result['time_to_first_frame_ms']}ms")
print(f"\n  💡 When bandwidth dropped (800→400 kbps), the player switched to 360p/480p")
print(f"     to avoid buffering. When it recovered, it switched back to 1080p.")
print(f"     The user never experienced a pause in playback!")

## 📊 Full Streaming Flow: GET /videos/:videoId

Let's put together the complete Video Service endpoint — what happens when a user clicks play.

In [ ]:
def get_video(video_id: str) -> dict:
    """
    Video Service: GET /videos/:videoId
    Returns metadata + manifest URL. Client handles streaming from there.
    """
    conn = get_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        SELECT id, title, description, status, duration_seconds, manifest_url
        FROM videos WHERE id = %s
    """, (video_id,))
    video = cur.fetchone()
    cur.close()
    conn.close()

    if not video or video["status"] != "ready":
        return {"error": "Video not found or not ready"}

    return {
        "video": {
            "id": video["id"],
            "title": video["title"],
            "description": video["description"],
            "duration": video["duration_seconds"],
        },
        "manifestUrl": video["manifest_url"],
    }


# Simulate: user clicks play
print("🖱️  User clicks play on 'Never Gonna Give You Up'\n")

# Step 1: GET /videos/:videoId
print("📌 Step 1: GET /videos/dQw4w9WgXcQ")
response = get_video("dQw4w9WgXcQ")
print(f"  Title: {response['video']['title']}")
print(f"  Duration: {response['video']['duration']}s")
print(f"  Manifest URL: {response['manifestUrl']}")

# Step 2: Client downloads master manifest
print(f"\n📌 Step 2: Client fetches master manifest")
qualities = parse_master_manifest(response["manifestUrl"])
print(f"  Available qualities:")
for q in qualities:
    print(f"    {q['resolution']} ({q['bandwidth'] // 1000} kbps)")

# Step 3: Client starts adaptive streaming
print(f"\n📌 Step 3: Client starts adaptive bitrate streaming")
print(f"  → Downloads segment 0 at chosen quality")
print(f"  → Starts playback immediately")
print(f"  → Prefetches upcoming segments")
print(f"  → Switches quality if bandwidth changes")
print(f"\n  💡 The server does ZERO video serving.")
print(f"     It returned metadata (~1KB) and a manifest URL.")
print(f"     All video data flows from S3/CDN directly to the client.")

## 🧮 What Does This Ladder Actually Cost?

Time for the back-of-envelope, and this is the one people most often get wrong.

The intuition "we store 4 qualities, so storage is **4x**" is badly off. Bitrate scales
roughly with pixel count, so the lower rungs are *tiny* compared to the top one:

```
1080p = 1920x1080 = 2,073,600 px  →  5.0 Mbps   (1.00x)
720p  = 1280x720  =   921,600 px  →  2.5 Mbps   (0.50x)
480p  =  854x480  =   409,920 px  →  1.0 Mbps   (0.20x)
360p  =  640x360  =   230,400 px  →  0.5 Mbps   (0.10x)
```

Add those up: **1.8x** the 1080p rendition, not 4x. Adding the bottom three rungs costs
80% more than storing 1080p alone, and buys you every viewer on a phone or a bad
connection.

We don't have to take that on faith — we just wrote all four renditions into MinIO. Let's
weigh them.

In [ ]:
# ── Measure what we actually stored, then extrapolate ──────────────────────
by_quality: dict[str, int] = {}
manifest_bytes = 0

for obj in minio_client.list_objects(BUCKET, prefix=f"{VIDEO_ID}/", recursive=True):
    if obj.object_name.endswith(".m3u8"):
        manifest_bytes += obj.size
    else:
        quality = obj.object_name.split("/")[1]
        by_quality[quality] = by_quality.get(quality, 0) + obj.size

top_rung = by_quality["1080p"]
ladder_total = sum(by_quality.values())

print(f"📦 Measured in S3 for {TOTAL_DURATION}s of video:\n")
for quality in QUALITIES:
    size = by_quality[quality]
    print(f"   {quality:<7} {size / 1024 / 1024:>7.2f} MB   "
          f"{size / top_rung:>5.2f}x of 1080p")
print(f"   {'-' * 38}")
print(f"   {'ladder':<7} {ladder_total / 1024 / 1024:>7.2f} MB   "
      f"{ladder_total / top_rung:>5.2f}x of 1080p")
print(f"   {'manifests':<7} {manifest_bytes:>7,} B   (rounding error — text files)")

naive = top_rung * len(QUALITIES)
print(f"\n   The 'four copies' guess would say {naive / 1024 / 1024:.2f} MB.")
print(f"   Reality is {ladder_total / 1024 / 1024:.2f} MB — "
      f"{naive / ladder_total:.1f}x smaller than the guess.")

# ── Scale it to the requirement: 1M uploads/day, 100M watches/day ──────────
UPLOADS_PER_DAY = 1_000_000
WATCHES_PER_DAY = 100_000_000
AVG_VIDEO_MINUTES = 10
AVG_WATCH_MINUTES = 5          # most viewers do not finish the video
MASTER_MBPS = 8                # what the user actually uploaded, pre-transcode
AVG_DELIVERED_MBPS = 2.5       # mix of 1080p desktop and 480p/720p mobile

ladder_mbps = sum(q["bandwidth"] for q in QUALITIES.values()) / 1e6
seconds = AVG_VIDEO_MINUTES * 60

master_gb = MASTER_MBPS * seconds / 8 / 1000
ladder_gb = ladder_mbps * seconds / 8 / 1000

print(f"\n{'=' * 62}")
print(f"📊 Storage — {UPLOADS_PER_DAY:,} uploads/day at {AVG_VIDEO_MINUTES} min each")
print(f"{'=' * 62}")
print(f"   uploaded master  @ {MASTER_MBPS:>4.1f} Mbps  = {master_gb:>5.2f} GB/video")
print(f"   full ladder      @ {ladder_mbps:>4.1f} Mbps  = {ladder_gb:>5.2f} GB/video"
      f"   ({ladder_gb / master_gb:.2f}x the master)")
print(f"\n   renditions : {UPLOADS_PER_DAY * ladder_gb / 1e6:>6.2f} PB/day"
      f"  →  {UPLOADS_PER_DAY * ladder_gb * 365 / 1e6:>6.0f} PB/year")
print(f"   + masters  : {UPLOADS_PER_DAY * master_gb / 1e6:>6.2f} PB/day"
      f"  →  {UPLOADS_PER_DAY * master_gb * 365 / 1e6:>6.0f} PB/year")
print("\n   Keeping the masters roughly doubles the bill. You keep them anyway —")
print("   the day you adopt AV1 you need the source to re-encode from, and")
print("   re-encoding from your own 1080p rendition compounds the artifacts.")
print("   Cold storage (Glacier) is the compromise: cheap to keep, slow to read,")
print("   which is fine because you only read them on a codec migration.")

# ── Egress: the number that decides the architecture ───────────────────────
watch_gb = AVG_DELIVERED_MBPS * AVG_WATCH_MINUTES * 60 / 8 / 1000
egress_pb_day = WATCHES_PER_DAY * watch_gb / 1e6
avg_gbps = egress_pb_day * 1e15 * 8 / 86400 / 1e9

print(f"\n{'=' * 62}")
print(f"📊 Egress — {WATCHES_PER_DAY:,} watches/day")
print(f"{'=' * 62}")
print(f"   per watch  : {AVG_WATCH_MINUTES} min @ {AVG_DELIVERED_MBPS} Mbps "
      f"= {watch_gb * 1000:.0f} MB")
print(f"   per day    : {egress_pb_day:.2f} PB")
print(f"   average    : {avg_gbps:,.0f} Gbps sustained")
print(f"   peak (~3x) : {avg_gbps * 3 / 1000:,.1f} Tbps")

print(f"""
   Read/write asymmetry: we ingest {UPLOADS_PER_DAY * master_gb / 1e6:.2f} PB/day and serve
   {egress_pb_day:.2f} PB/day — about {egress_pb_day / (UPLOADS_PER_DAY * master_gb / 1e6):.0f}x more out than in. Every design decision
   should be optimising the read path.

   And at a typical cloud egress price of $0.05/GB, {egress_pb_day:.1f} PB/day is
   ${egress_pb_day * 1e6 * 0.05 / 1e6:.2f}M/day — ${egress_pb_day * 1e6 * 0.05 * 365 / 1e6:,.0f}M/year — if it all left S3.

   That single number is why YouTube built Google Global Cache and Netflix built
   Open Connect: at this volume you stop renting bandwidth and start shipping
   your own boxes into ISPs. The CDN in our architecture diagram is not a
   latency optimisation. It is the business model.""")

## 🧹 Cleanup

In [ ]:
# Clean up processed segments (keep video metadata for future labs)
from minio.deleteobjects import DeleteObject

objects = list(minio_client.list_objects(BUCKET, prefix=f"{VIDEO_ID}/", recursive=True))
if objects:
    delete_list = [DeleteObject(obj.object_name) for obj in objects]
    errors = list(minio_client.remove_objects(BUCKET, delete_list))
    print(f"✅ Cleaned {len(delete_list)} objects from processed-videos/{VIDEO_ID}/")
else:
    print("✅ Already clean.")